# Multi-Layer Perceptron (MLP) Backpropagation from Scratch

This notebook implements a complete 3-layer feedforward neural network (MLP) from scratch using only raw NumPy. We will implement forward propagation, cross-entropy loss, backpropagation matrix calculus, and batch gradient descent, strictly utilizing **Andrew Ng's vectorized notation**.

## 1. Mathematical Summary of the Vectorized Equations

Let $m$ be the number of training examples, $n_x$ be the input feature size, and $L$ be the output layer index.

### Forward Propagation
For layer $l = 1 \dots L$:
- $Z^{[l]} = W^{[l]} A^{[l-1]} + b^{[l]}$
- $A^{[l]} = g^{[l]}(Z^{[l]})$

Where $A^{[0]} = X$ is the input matrix of shape $(n_x, m)$ (each column is a training sample).

### Backward Propagation (Vectorized Gradients)
For the output layer $L$ using Softmax activation and Categorical Cross-Entropy Loss:
- $dZ^{[L]} = A^{[L]} - Y$ (where $Y$ is the one-hot target matrix of shape $(class\_count, m)$)

For hidden layers $l = L-1 \dots 1$:
- $dA^{[l]} = W^{[l+1]T} dZ^{[l+1]}$
- $dZ^{[l]} = dA^{[l]} \odot g^{[l]\prime}(Z^{[l]})$

To update parameters:
- $dW^{[l]} = \frac{1}{m} dZ^{[l]} A^{[l-1]T}$
- $db^{[l]} = \frac{1}{m} \sum_{columns} dZ^{[l]} = \frac{1}{m} \text{np.sum}(dZ^{[l]}, \text{axis}=1, \text{keepdims}=\text{True})$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

# 1. Generate non-linear classification dataset (Moons)
np.random.seed(42)
X_raw, y_raw = make_moons(n_samples=400, noise=0.2, random_state=42)

# Format inputs as Andrew Ng notation: shape (n_x, m) -> (2, 400)
X = X_raw.T
m = X.shape[1]

# One-hot encode targets: shape (2, 400)
Y = np.zeros((2, m))
Y[y_raw, np.arange(m)] = 1.0

print(f"Inputs X shape: {X.shape} (n_x={X.shape[0]}, m={X.shape[1]})")
print(f"Targets Y shape: {Y.shape}")

## 2. Activation Functions and Derivatives

In [ ]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return np.where(Z > 0, 1.0, 0.0)

def softmax(Z):
    # Subtract max for numerical stability (prevents overflow)
    shift_Z = Z - np.max(Z, axis=0, keepdims=True)
    exp_Z = np.exp(shift_Z)
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

## 3. Initialize Model Parameters (He/Kaiming Initialization)

In [ ]:
def initialize_parameters(layer_dims):
    # layer_dims = [input_dim, hidden_dim, output_dim]
    np.random.seed(42)
    parameters = {}
    L = len(layer_dims) - 1
    
    for l in range(1, L + 1):
        # He Initialization: Var(W) = 2/n_in
        parameters[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2.0 / layer_dims[l-1])
        parameters[f'b{l}'] = np.zeros((layer_dims[l], 1))
        
        print(f"Layer {l}: W{l} shape: {parameters[f'W{l}'].shape} | b{l} shape: {parameters[f'b{l}'].shape}")
        
    return parameters

## 4. Forward Propagation & Loss

In [ ]:
def forward_propagation(X, parameters):
    # 3-Layer MLP forward pass
    W1, b1 = parameters['W1'], parameters['b1']
    W2, b2 = parameters['W2'], parameters['b2']
    
    # Layer 1 (ReLU hidden layer)
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    
    # Layer 2 (Softmax output layer)
    Z2 = np.dot(W2, A1) + b2
    A2 = softmax(Z2)
    
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def compute_loss(A2, Y):
    # Categorical Cross-Entropy Loss
    m = Y.shape[1]
    # Add small epsilon to prevent log(0) numerical instability
    loss = -1.0 / m * np.sum(Y * np.log(A2 + 1e-15))
    return float(np.squeeze(loss))

## 5. Backward Propagation

In [ ]:
def backward_propagation(X, Y, cache, parameters):
    m = X.shape[1]
    A1, A2 = cache['A1'], cache['A2']
    Z1 = cache['Z1']
    W2 = parameters['W2']
    
    # 1. Output layer error gradient (Softmax + Cross-Entropy)
    dZ2 = A2 - Y
    dW2 = 1.0 / m * np.dot(dZ2, A1.T)
    db2 = 1.0 / m * np.sum(dZ2, axis=1, keepdims=True)
    
    # 2. Hidden layer error gradient (Backpropagated to ReLU)
    dA1 = np.dot(W2.T, dZ2)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = 1.0 / m * np.dot(dZ1, X.T)
    db1 = 1.0 / m * np.sum(dZ1, axis=1, keepdims=True)
    
    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

## 6. Training Loop

In [ ]:
layer_dims = [2, 10, 2] # 2 inputs, 10 hidden units, 2 output classes
parameters = initialize_parameters(layer_dims)

learning_rate = 0.3
epochs = 1000
losses = []

print("\n--- Training Loop Starting ---")
for epoch in range(epochs):
    A2, cache = forward_propagation(X, parameters)
    loss = compute_loss(A2, Y)
    grads = backward_propagation(X, Y, cache, parameters)
    
    # Parameter update step
    parameters['W1'] -= learning_rate * grads['dW1']
    parameters['b1'] -= learning_rate * grads['db1']
    parameters['W2'] -= learning_rate * grads['dW2']
    parameters['b2'] -= learning_rate * grads['db2']
    
    if epoch % 100 == 0:
        losses.append(loss)
        # Calculate accuracy
        preds = np.argmax(A2, axis=0)
        acc = np.mean(preds == y_raw) * 100.0
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Accuracy: {acc:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(np.arange(0, epochs, 100), losses, marker='o', color='#339af0')
plt.title("NumPy MLP Training Loss Convergence")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()